In [1]:
import pandas as pd

# Load data
influenza_NY_data = pd.read_csv("Influenza_NY.csv")
coords_df = pd.read_csv("NY Counties with Coords.csv")

# Clean column names
influenza_NY_data.columns = influenza_NY_data.columns.str.strip()
coords_df.columns = coords_df.columns.str.strip().str.title()  


# Clean County names
influenza_NY_data["County"] = influenza_NY_data["County"].str.strip().str.upper()
coords_df["County"] = coords_df["County"].str.strip().str.upper()

# Fix mismatches between lattitude and longitude data and influenza data
mapping = {
    "KINGS": "KINGS (BROOKLYN)",
    "NEW YORK": "NEW YORK (MANHATTAN)",
    "RICHMOND": "RICHMOND (STATEN ISLAND)",
    "ST LAWRENCE": "ST. LAWRENCE"
}

influenza_NY_data["County"] = influenza_NY_data["County"].replace(mapping)

# Remove duplicates in coords
coords_df = coords_df.drop_duplicates(subset="County")

# Merge
influenza_NY_data = influenza_NY_data.merge(
    coords_df,
    on="County",
    how="left"
)

# Validate merge
missing_lat = influenza_NY_data["Latitude"].isna().sum()
missing_lon = influenza_NY_data["Longitude"].isna().sum()

print("Missing Latitude:", missing_lat)
print("Missing Longitude:", missing_lon)

influenza_NY_data = influenza_NY_data.drop(influenza_NY_data.columns[0],axis='columns') #first column is useless
influenza_NY_data['Prob_infected'] = influenza_NY_data['Infected']/influenza_NY_data['Population'] # Probability of being infected (Weekly)
influenza_NY_data['County_Density']=influenza_NY_data['Population']/influenza_NY_data['Area'] # Density of population of the area

#result
print(influenza_NY_data[['County', 'Latitude', 'Longitude','Disease','Infected','Prob_infected','County_Density','Week Ending Date']].head(10))



Missing Latitude: 0
Missing Longitude: 0
   County  Latitude  Longitude                Disease  Infected  \
0  ALBANY   42.6006   -73.9772            INFLUENZA_A         4   
1  ALBANY   42.6006   -73.9772            INFLUENZA_B         0   
2  ALBANY   42.6006   -73.9772  INFLUENZA_UNSPECIFIED         0   
3  ALBANY   42.6006   -73.9772            INFLUENZA_A        11   
4  ALBANY   42.6006   -73.9772            INFLUENZA_B         0   
5  ALBANY   42.6006   -73.9772  INFLUENZA_UNSPECIFIED         0   
6  ALBANY   42.6006   -73.9772            INFLUENZA_A        27   
7  ALBANY   42.6006   -73.9772            INFLUENZA_B         0   
8  ALBANY   42.6006   -73.9772  INFLUENZA_UNSPECIFIED         0   
9  ALBANY   42.6006   -73.9772            INFLUENZA_A        86   

   Prob_infected  County_Density Week Ending Date  
0       0.000013      582.162575       10/10/2009  
1       0.000000      582.162575       10/10/2009  
2       0.000000      582.162575       10/10/2009  
3       0.000

In [2]:
import openmeteo_requests
import requests_cache
from retry_requests import retry

# Setup the Open-Meteo API client with cache and retry on error
cache_session = requests_cache.CachedSession('.cache', expire_after=-1)
retry_session = retry(cache_session, retries=5, backoff_factor=0.2)
openmeteo = openmeteo_requests.Client(session=retry_session)

In [ ]:
import pandas as pd
from datetime import timedelta
import time 


# Setup Data
influenza_NY_data['Ending Date'] = pd.to_datetime(influenza_NY_data['Week Ending Date']).dt.tz_localize(None)
\
# Clean coordinates at the source to prevent "ghost" duplicates
influenza_NY_data['Latitude'] = pd.to_numeric(influenza_NY_data['Latitude']).round(3)
influenza_NY_data['Longitude'] = pd.to_numeric(influenza_NY_data['Longitude']).round(3)

# Then get unique locations
unique_locations = influenza_NY_data[['Latitude', 'Longitude']].drop_duplicates().reset_index(drop=True)

# Define the date window
overall_start = (influenza_NY_data['Ending Date'].min() - timedelta(days=7)).strftime('%Y-%m-%d')
overall_end = influenza_NY_data['Ending Date'].max().strftime('%Y-%m-%d')


# Batched API Calls
daily_url = "https://archive-api.open-meteo.com/v1/archive"
all_weather_results = []

# Smaller batches, to not go over API rate limit 
batch_size = 10 

for start_idx in range(0, len(unique_locations), batch_size):
    end_idx = min(start_idx + batch_size, len(unique_locations))
    batch = unique_locations.iloc[start_idx:end_idx]
    
    print(f"Requesting counties {start_idx} through {end_idx}...")
    
    params = {
        "latitude": batch['Latitude'].tolist(),
        "longitude": batch['Longitude'].tolist(),
        "start_date": overall_start,
        "end_date": overall_end,
        "models": "era5_land",  # more specific model for historical data
        "daily": [
            "temperature_2m_mean", 
            "relative_humidity_2m_mean", 
            "dew_point_2m_mean", 
            "precipitation_sum", 
            "wind_speed_10m_max"
        ],
        "timezone": "auto",
        "temperature_unit": "fahrenheit",
        "wind_speed_unit": "mph",
        "precipitation_unit": "inch"
    }

    try:
        responses = openmeteo.weather_api(daily_url, params=params)

        for i, response in enumerate(responses):
            lat = batch.iloc[i]['Latitude']
            lon = batch.iloc[i]['Longitude']
            daily = response.Daily()
            
            loc_daily = pd.DataFrame({
                "date": pd.date_range(
                    start=pd.to_datetime(daily.Time(), unit="s", utc=True),
                    periods=daily.Variables(0).ValuesAsNumpy().size,
                    freq=pd.Timedelta(seconds=daily.Interval())
                ).tz_localize(None),
                "temp": daily.Variables(0).ValuesAsNumpy(),
                "humidity": daily.Variables(1).ValuesAsNumpy(),
                "dew_point": daily.Variables(2).ValuesAsNumpy(),
                "precip": daily.Variables(3).ValuesAsNumpy(),
                "wind_speed": daily.Variables(4).ValuesAsNumpy()
            })
            
            loc_daily['Latitude'], loc_daily['Longitude'] = lat, lon
            all_weather_results.append(loc_daily)
            
    except Exception as e:
        print(f"Error at batch {start_idx}: {e}")
        print("Waiting 10 seconds and trying to continue...")
        time.sleep(10) # Longer wait if we hit a snag
        continue

    # Pausing for 2 seconds between every batch request to prevent rate limit errors
    time.sleep(2)

final_weather_df = pd.concat(all_weather_results)
print("All batches downloaded successfully!")


# Calculate Rolling Averages ---
final_weather_df = final_weather_df.sort_values(['Latitude', 'Longitude', 'date'])
weather_cols = ['temp', 'humidity', 'dew_point', 'precip', 'wind_speed']

rolling_weather = (
    final_weather_df.groupby(['Latitude', 'Longitude'])[weather_cols] 
    .rolling(window=7, min_periods=1) 
    .mean()
    .reset_index()
)

# Rename before merging to avoid confusion with daily values
rename_dict = {col: f"avg_{col}_prior_week" for col in weather_cols}
rolling_weather.rename(columns=rename_dict, inplace=True)


# --- DEBUG CHECK ---
print("Columns currently in rolling_weather:", rolling_weather.columns.tolist())
print(rolling_weather[list(rename_dict.values())].count())


# --- STEP #3: PLACEHOLDER REMOVAL (FIXED) ---
# 1. Calculate how many unique values exist for temperature per coordinate
# We use .nunique() directly on the GroupBy object
variance_check = rolling_weather.groupby(['Latitude', 'Longitude'])['avg_temp_prior_week'].nunique()

# 2. Identify the "Bad" coordinates (where only 1 unique value exists, usually 60.0)
bad_coords = variance_check[variance_check == 1].index

if not bad_coords.empty:
    print(f"Detected {len(bad_coords)} locations with zero variance. Converting to NaN...")
    
    # 3. Create a list of the columns we want to wipe
    # Use your existing weather_cols list to generate the 'avg_...' names
    cols_to_wipe = [f"avg_{col}_prior_week" for col in weather_cols]
    
    # 4. Use .set_index temporarily to make wiping by coordinates easy
    rolling_weather = rolling_weather.set_index(['Latitude', 'Longitude'])
    
    # Set the bad rows to NaN
    rolling_weather.loc[bad_coords, cols_to_wipe] = np.nan
    
    # Reset index to bring Lat/Lon back as columns for your merge
    rolling_weather = rolling_weather.reset_index()
# --- END STEP #3 ---


# Standardize Dates
rolling_weather['date'] = pd.Series(final_weather_df['date'].values).dt.normalize().values
influenza_NY_data['Ending Date'] = pd.to_datetime(influenza_NY_data['Ending Date']).dt.normalize()


# Standardize Lat/Lon to 3 decimal places to ensure merge compatibility
for df in [influenza_NY_data, rolling_weather]:
    df['Latitude'] = pd.to_numeric(df['Latitude']).round(3)
    df['Longitude'] = pd.to_numeric(df['Longitude']).round(3)



# Identify counties that are 'dead' (zero variance)
variance_counts = rolling_weather.groupby(['Latitude', 'Longitude'])['avg_temp_prior_week'].nunique()
dead_locations = variance_counts[variance_counts <= 1].index

if not dead_locations.empty:
    print(f"Cleaning {len(dead_locations)} 'ghost' locations...")
    rolling_weather = rolling_weather.set_index(['Latitude', 'Longitude'])
    
    # Wipe all weather columns for these counties
    cols_to_fix = [f"avg_{col}_prior_week" for col in weather_cols]
    rolling_weather.loc[dead_locations, cols_to_fix] = np.nan
    
    rolling_weather = rolling_weather.reset_index()



# Single Merge
final_table = pd.merge(
    influenza_NY_data, 
    rolling_weather, 
    left_on=['Latitude', 'Longitude', 'Ending Date'], 
    right_on=['Latitude', 'Longitude', 'date'], 
    how='left'
)

# Cleaning data and filling gaps
missing_after_merge = final_table[final_table['avg_temp_prior_week'].isna()]['County'].unique()
if len(missing_after_merge) > 0:
    print(f"Filling gaps for: {missing_after_merge}")
    # Backfill/Forward fill gaps within counties
    final_table = final_table.sort_values(['County', 'Ending Date'])
    new_cols = list(rename_dict.values())
    final_table[new_cols] = final_table.groupby('County')[new_cols].ffill().bfill()

# Final export cleanup
if 'date' in final_table.columns:
    final_table = final_table.drop(columns=['date'])



print("Success! Dataset exported.")
print(final_table[['County', 'Ending Date', 'avg_temp_prior_week', 'avg_precip_prior_week']].head())

Requesting counties 0 through 10...
Requesting counties 10 through 20...
Requesting counties 20 through 30...
Requesting counties 30 through 40...
Requesting counties 40 through 50...
Requesting counties 50 through 60...
Requesting counties 60 through 62...
All batches downloaded successfully!
Columns currently in rolling_weather: ['Latitude', 'Longitude', 'level_2', 'avg_temp_prior_week', 'avg_humidity_prior_week', 'avg_dew_point_prior_week', 'avg_precip_prior_week', 'avg_wind_speed_prior_week']
avg_temp_prior_week          213156
avg_humidity_prior_week      213156
avg_dew_point_prior_week     213156
avg_precip_prior_week             0
avg_wind_speed_prior_week         0
dtype: int64
Success! Dataset exported.
   County Ending Date  avg_temp_prior_week  avg_precip_prior_week
0  ALBANY  2009-10-10            52.136235                    NaN
1  ALBANY  2009-10-10            52.136235                    NaN
2  ALBANY  2009-10-10            52.136235                    NaN
3  ALBANY  200

In [26]:
# Setup Data
influenza_NY_data['Ending Date'] = pd.to_datetime(influenza_NY_data['Week Ending Date']).dt.tz_localize(None)
\
# Clean coordinates at the source to prevent "ghost" duplicates
influenza_NY_data['Latitude'] = pd.to_numeric(influenza_NY_data['Latitude']).round(3)
influenza_NY_data['Longitude'] = pd.to_numeric(influenza_NY_data['Longitude']).round(3)

# Then get unique locations
unique_locations = influenza_NY_data[['Latitude', 'Longitude']].drop_duplicates().reset_index(drop=True)

# Define the date window
overall_start = (influenza_NY_data['Ending Date'].min() - timedelta(days=7)).strftime('%Y-%m-%d')
overall_end = influenza_NY_data['Ending Date'].max().strftime('%Y-%m-%d')


# Batched API Calls
daily_url = "https://archive-api.open-meteo.com/v1/archive"
all_atmos_results = []

# Smaller batches, to not go over API rate limit 
batch_size = 10 

for start_idx in range(0, len(unique_locations), batch_size):
    end_idx = min(start_idx + batch_size, len(unique_locations))
    batch = unique_locations.iloc[start_idx:end_idx]
    
    print(f"Requesting counties {start_idx} through {end_idx}...")
    
    params = {
        "latitude": batch['Latitude'].tolist(),
        "longitude": batch['Longitude'].tolist(),
        "start_date": overall_start,
        "end_date": overall_end,
        "models": "era5",  
        "daily": ["precipitation_sum", "wind_speed_10m_max"], 
        "timezone": "auto",
        "wind_speed_unit": "mph",
        "precipitation_unit": "inch"
    }

    try:
        responses = openmeteo.weather_api(daily_url, params=params)

        for i, response in enumerate(responses):
            lat = batch.iloc[i]['Latitude']
            lon = batch.iloc[i]['Longitude']
            daily = response.Daily()
            
            loc_daily = pd.DataFrame({
                "date": pd.date_range(
                    start=pd.to_datetime(daily.Time(), unit="s", utc=True),
                    periods=daily.Variables(0).ValuesAsNumpy().size,
                    freq=pd.Timedelta(seconds=daily.Interval())
                ).tz_localize(None),
                "precip": daily.Variables(0).ValuesAsNumpy(),
                "wind_speed": daily.Variables(1).ValuesAsNumpy()
            })
            
            loc_daily['Latitude'], loc_daily['Longitude'] = lat, lon
            all_atmos_results.append(loc_daily)
            
    except Exception as e:
        print(f"Error at batch {start_idx}: {e}")
        time.sleep(10) # Longer wait if we hit a snag
        continue

    # Pausing for 2 seconds between every batch request to prevent rate limit errors
    time.sleep(2)

if len(all_atmos_results) > 0:
    df_atmos_raw = pd.concat(all_atmos_results)
    print("All batches downloaded successfully!")
else:
    print("Error: No data was collected. Check your API connection.")


# Calculate Rolling Averages ---
weather_cols = ['precip', 'wind_speed']

rolling_atmos = (
    df_atmos_raw.sort_values(['Latitude', 'Longitude', 'date'])
    .groupby(['Latitude', 'Longitude'])[weather_cols + ['date']]
    .rolling(window=7, min_periods=1, on='date') 
    .mean()
    .reset_index()
)

# Rename before merging to avoid confusion with daily values
rename_dict = {col: f"avg_{col}_prior_week" for col in weather_cols}
rolling_atmos.rename(columns=rename_dict, inplace=True)

# Normalize dates so they match perfectly (removes time components)
rolling_atmos['date'] = pd.to_datetime(rolling_atmos['date']).dt.normalize()
influenza_NY_data['Ending Date'] = pd.to_datetime(influenza_NY_data['Ending Date']).dt.normalize()

# Round coordinates again just to be 100% sure they match
for df in [influenza_NY_data, rolling_atmos]:
    df['Latitude'] = pd.to_numeric(df['Latitude']).round(3)
    df['Longitude'] = pd.to_numeric(df['Longitude']).round(3)

# Single Merge
final_atmos_table = pd.merge(
    influenza_NY_data, 
    rolling_atmos, 
    left_on=['Latitude', 'Longitude', 'Ending Date'], 
    right_on=['Latitude', 'Longitude', 'date'], 
    how='left'
)


# Final export cleanup
if 'date' in final_atmos_table.columns:
    final_atmos_table = final_atmos_table.drop(columns=['date'])



print("Success! Dataset exported.")
print(final_atmos_table[['County', 'Ending Date','avg_precip_prior_week', 'avg_wind_speed_prior_week']].head())

Requesting counties 0 through 10...
Requesting counties 10 through 20...
Requesting counties 20 through 30...
Requesting counties 30 through 40...
Requesting counties 40 through 50...
Requesting counties 50 through 60...
Requesting counties 60 through 62...
All batches downloaded successfully!
Success! Dataset exported.
   County Ending Date  avg_precip_prior_week  avg_wind_speed_prior_week
0  ALBANY  2009-10-10               0.082677                  11.513748
1  ALBANY  2009-10-10               0.082677                  11.513748
2  ALBANY  2009-10-10               0.082677                  11.513748
3  ALBANY  2009-10-17               0.042745                   7.608064
4  ALBANY  2009-10-17               0.042745                   7.608064


In [32]:
# --- STEP 1: PRE-CLEANING (The "Pruning" Phase) ---

# We only want the weather stats and the keys. 
# This prevents 'level_2', 'index', or old 'precip' columns from causing _x/_y suffixes.

cols_to_keep_weather = ['Latitude', 'Longitude', 'date', 'avg_temp_prior_week', 'avg_humidity_prior_week', 'avg_dew_point_prior_week']
cols_to_keep_atmos = ['Latitude', 'Longitude', 'date', 'avg_precip_prior_week', 'avg_wind_speed_prior_week']

# Filter the rolling dataframes to ONLY these columns
rolling_weather_clean = rolling_weather[cols_to_keep_weather].copy()
rolling_atmos_clean = rolling_atmos[cols_to_keep_atmos].copy()

# --- STEP 2: COORDINATE STANDARDIZATION ---

# Rounding to 3 decimals ensures the merge keys match perfectly across all three dataframes
for df in [influenza_NY_data, rolling_weather_clean, rolling_atmos_clean]:
    df['Latitude'] = pd.to_numeric(df['Latitude']).round(3)
    df['Longitude'] = pd.to_numeric(df['Longitude']).round(3)
    # Ensure all join dates are normalized (no time component)
    if 'date' in df.columns:
        df['date'] = pd.to_datetime(df['date']).dt.normalize()

# Ensure the main influenza date is also normalized
influenza_NY_data['Ending Date'] = pd.to_datetime(influenza_NY_data['Ending Date']).dt.normalize()

# --- STEP 3: THE SEQUENTIAL MERGE ---

# Merge 1: Influenza + Land Data (Temp/Humid)
step1 = pd.merge(
    influenza_NY_data, 
    rolling_weather_clean, 
    left_on=['Latitude', 'Longitude', 'Ending Date'], 
    right_on=['Latitude', 'Longitude', 'date'], 
    how='left'
).drop(columns=['date'], errors='ignore')

# Merge 2: Result + Atmos Data (Precip/Wind)
final_master_table = pd.merge(
    step1, 
    rolling_atmos_clean, 
    left_on=['Latitude', 'Longitude', 'Ending Date'], 
    right_on=['Latitude', 'Longitude', 'date'], 
    how='left'
).drop(columns=['date'], errors='ignore')

# --- STEP 4: FINAL VALIDATION ---

print("Master Table Columns:", final_master_table.columns.tolist())

# Check if we have any NaNs in our new weather columns
print("\nMissing Values Count:")
print(final_master_table[['avg_temp_prior_week', 'avg_precip_prior_week']].isna().sum())

final_master_table.to_csv("NY_Flu_Weather_Final_Weekly.csv", index=False)

final_master_table.head()

Master Table Columns: ['County', 'Year', 'Month', 'Season', 'Region', 'Week', 'Week Ending Date', 'Disease', 'Infected', 'Avg household size', 'Area', 'Population', 'Under_18', '18-24', '25-44', '45-64', 'Above_65', 'Median_age', 'Medianfamilyincome', 'Number_households', 'Beds_adult_facility_care', 'Beds_hospital', 'County_Served_hospital', 'Service_hospital', 'Discharges_Other_Hospital_intervention', 'Discharges_Respiratory_system_interventions', 'Total_Charge_Other_Hospital_intervention', 'Total_Charge_Respiratory_system_interventions', 'Unemp_rate', 'Latitude', 'Longitude', 'Prob_infected', 'County_Density', 'Ending Date', 'avg_temp_prior_week', 'avg_humidity_prior_week', 'avg_dew_point_prior_week', 'avg_precip_prior_week', 'avg_wind_speed_prior_week']

Missing Values Count:
avg_temp_prior_week      0
avg_precip_prior_week    0
dtype: int64


,County,Year,Month,Season,Region,Week,Week Ending Date,Disease,Infected,Avg household size,...,Latitude,Longitude,Prob_infected,County_Density,Ending Date,avg_temp_prior_week,avg_humidity_prior_week,avg_dew_point_prior_week,avg_precip_prior_week,avg_wind_speed_prior_week
0,ALBANY,2009,10,2009-2010,CAPITAL DISTRICT,40,10/10/2009,INFLUENZA_A,4,2.3,...,42.601,-73.977,0.000013,582.162575,2009-10-10,52.136235,75.422893,44.082307,0.082677,11.513748
1,ALBANY,2009,10,2009-2010,CAPITAL DISTRICT,40,10/10/2009,INFLUENZA_B,0,2.3,...,42.601,-73.977,0.000000,582.162575,2009-10-10,52.136235,75.422893,44.082307,0.082677,11.513748
2,ALBANY,2009,10,2009-2010,CAPITAL DISTRICT,40,10/10/2009,INFLUENZA_UNSPECIFIED,0,2.3,...,42.601,-73.977,0.000000,582.162575,2009-10-10,52.136235,75.422893,44.082307,0.082677,11.513748
3,ALBANY,2009,10,2009-2010,CAPITAL DISTRICT,41,10/17/2009,INFLUENZA_A,11,2.3,...,42.601,-73.977,0.000036,582.162575,2009-10-17,39.730165,68.109046,29.425700,0.042745,7.608064
4,ALBANY,2009,10,2009-2010,CAPITAL DISTRICT,41,10/17/2009,INFLUENZA_B,0,2.3,...,42.601,-73.977,0.000000,582.162575,2009-10-17,39.730165,68.109046,29.425700,0.042745,7.608064


In [31]:
# check the weather data for a single location across time
sample_lat = rolling_weather['Latitude'].iloc[0]
sample_lon = rolling_weather['Longitude'].iloc[0]

sample_check = rolling_weather[(rolling_weather['Latitude'] == sample_lat) & 
                               (rolling_weather['Longitude'] == sample_lon)]

print(f"Total weeks for this location: {len(sample_check)}")
print(f"Unique temperatures for this location: {sample_check['avg_temp_prior_week'].nunique()}")

Total weeks for this location: 3438
Unique temperatures for this location: 3427
